In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime, timedelta
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import WebDriverException
import time
import re
from bs4 import BeautifulSoup
import pandas as pd
import dateparser

In [ ]:
# parse the dates

def parse_flexible_date(text):
    months_fr = {
        "janvier": "01", "février": "02", "fevrier": "02", "mars": "03", "avril": "04",
        "mai": "05", "juin": "06", "juillet": "07", "août": "08", "aout": "08",
        "septembre": "09", "octobre": "10", "novembre": "11", "décembre": "12", "decembre": "12"
    }
    text = text.lower()
    text = text.replace("é", "e").replace("è", "e").replace("ê", "e").replace("à", "a").strip()

    for fmt in ("%d-%m-%Y", "%d/%m/%Y", "%d.%m.%Y"):
        try:
            return datetime.strptime(text, fmt)
        except:
            continue

    match = re.search(r"(\d{1,2})\s+([a-zéûôî]+)\s+(\d{4})", text)
    if match:
        day, month_name, year = match.groups()
        month_num = months_fr.get(month_name)
        if month_num:
            try:
                return datetime.strptime(f"{day}-{month_num}-{year}", "%d-%m-%Y")
            except:
                return None
    return None

def extract_expiration_date(soup):

    full_text = soup.get_text(separator=" ", strip=True).lower()
    candidates = re.findall( r"(au plus tard(?: le)?|before the)\s+([0-9]{1,2}[\s/-]?(?:\w+)?[\s/-]?[0-9]{2,4}(?:\s?[0-9]{2}:[0-9]{2}(:[0-9]{2})?)?)",full_text, flags=re.IGNORECASE)

    for _, date_str in candidates:
        date_str = date_str.replace("/", "-").replace(".", "-")
        parsed = parse_flexible_date(date_str)
        if parsed:
            return parsed
    return None

In [ ]:
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"

options = webdriver.ChromeOptions()
options.add_argument(f"user-agent={user_agent}")

driver = webdriver.Chrome(options=options)

In [ ]:
# national calls for tenders

try:
    driver.get("LINK") #add link
except WebDriverException as e:
    print(f"Error while loading the page: {e}")
    print("Waiting before retrying...")
    time.sleep(10) 
wait = WebDriverWait(driver, 10)

In [ ]:
# scraping

today = datetime.now()
stop = False
tenders =[]

while True:
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li.list-group-item")))
    blocks = driver.find_elements(By.CSS_SELECTOR, "li.list-group-item")

    for i in range(len(blocks)):
        try:
            blocks = driver.find_elements(By.CSS_SELECTOR, "li.list-group-item")
            block = blocks[i]

            date_pub_text = block.find_element(By.XPATH, ".//div[contains(text(), 'Publié le')]/following-sibling::div").text.strip()

            date_pub = dateparser.parse(date_pub_text, settings={"DATE_ORDER": "DMY"})
            if not date_pub:
                continue
            
            if (today - date_pub).days > 30:
                print(f"Too old: {date_pub.strftime('%d-%m-%Y')}")
                stop = True
                break

            title = block.find_element(By.CSS_SELECTOR, "div.text-left strong").get_attribute("title").strip()
            publish_date = date_pub.strftime('%d-%m-%Y')
            link = block.find_element(By.XPATH, ".//a[contains(text(), 'Détails')]").get_attribute("href")    
            if link:
                driver.get(link)
                time.sleep(2)
                html = driver.page_source
                soup = BeautifulSoup(html, "html.parser")    
                expiration_date = None
                trs = soup.select("tbody tr")

                if len(trs) >= 2:
                    text = trs[1].get_text(separator=" ", strip=True).lower()
                    match = re.search(r"au plus tard(?: le)?\s+(\d{1,2}[/-]\d{1,2}[/-]\d{4})", text)
                    if match:
                        date_str = match.group(1).replace("/", "-")
                        try:
                            expiration_date = datetime.strptime(date_str, "%d-%m-%Y")
                        except:
                            pass

                if not expiration_date:
                    full_text = soup.get_text(separator=" ", strip=True).lower()
                    match = re.search(r"au plus tard(?: le)?\s+(\d{1,2}[/-]\d{1,2}[/-]\d{4})", full_text)
                    if not match:
                        match = re.search(r"before the\s+(\d{1,2}[/-]\d{1,2}[/-]\d{4})", full_text)
                    if match:
                        date_str = match.group(1).replace("/", "-")
                        try:
                            expiration_date = datetime.strptime(date_str, "%d-%m-%Y")
                        except:
                            pass    

                tenders.append({
                        "title": title,
                        "publication_date": publish_date,
                        "expiration_date": expiration_date.strftime('%d-%m-%Y') if expiration_date else None,
                        "link": link,
                        "country": "Cameroon"
                    })
                
                driver.back()
                wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li.list-group-item")))

        except Exception as e:
            print(f"Error on a block: {e}")

    if stop:
        break

    try:
      
        next_btn = driver.find_element(By.CSS_SELECTOR, "ul.pagination li.page-item a[rel='next']")
        if next_btn.is_enabled() and next_btn.is_displayed():
            next_btn.click()
            print("Next page")
            time.sleep(3) 
        else:
            print("No active next page")
            break
    except Exception:
        print("End of pagination")
        break

In [ ]:
# international calls for tenders

try:
    driver.get("https://www.armp.cm/filtres?type=avis&val=4")
except WebDriverException as e:
    print(f"Error while loading the page: {e}")
    print("Waiting before retrying...")
    time.sleep(10) 
wait = WebDriverWait(driver, 10)

In [ ]:
# scraping

today = datetime.now()
stop = False

while True:
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li.list-group-item")))
    blocks = driver.find_elements(By.CSS_SELECTOR, "li.list-group-item")

    for i in range(len(blocks)):
        try:
            blocks = driver.find_elements(By.CSS_SELECTOR, "li.list-group-item")
            block = blocks[i]

            date_pub_text = block.find_element(By.XPATH, ".//div[contains(text(), 'Publié le')]/following-sibling::div").text.strip()

            date_pub = dateparser.parse(date_pub_text, settings={"DATE_ORDER": "DMY"})
            if not date_pub:
                continue
            
            if (today - date_pub).days > 30:
                print(f"Too old: {date_pub.strftime('%d-%m-%Y')}")
                stop = True
                break

            title = block.find_element(By.CSS_SELECTOR, "div.text-left strong").get_attribute("title").strip()
            publish_date = date_pub.strftime('%d-%m-%Y')
            link = block.find_element(By.XPATH, ".//a[contains(text(), 'Détails')]").get_attribute("href")

            if link:
                driver.get(link)
                time.sleep(2)
                html = driver.page_source
                soup = BeautifulSoup(html, "html.parser")

                expiration_date = None
                trs = soup.select("tbody tr")
                if len(trs) >= 2:
                    text = trs[1].get_text(separator=" ", strip=True).lower()
                    match = re.search(r"au plus tard(?: le)?\s+(\d{1,2}[/-]\d{1,2}[/-]\d{4})", text)
                    if match:
                        date_str = match.group(1).replace("/", "-")
                        try:
                            expiration_date = datetime.strptime(date_str, "%d-%m-%Y")
                        except:
                            pass

                if not expiration_date:
                    full_text = soup.get_text(separator=" ", strip=True).lower()
                    match = re.search(r"au plus tard(?: le)?\s+(\d{1,2}[/-]\d{1,2}[/-]\d{4})", full_text)
                    if not match:
                        match = re.search(r"before the\s+(\d{1,2}[/-]\d{1,2}[/-]\d{4})", full_text)
                    if match:
                        date_str = match.group(1).replace("/", "-")
                        try:
                            expiration_date = datetime.strptime(date_str, "%d-%m-%Y")
                        except:
                            pass

                tenders.append({
                        "title": title,
                        "publication_date": publish_date,
                        "expiration_date": expiration_date.strftime('%d-%m-%Y') if expiration_date else None,
                        "link": link,
                        "country": "Cameroon"
                    })
                
                driver.back()
                wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li.list-group-item")))

        except Exception as e:
            print(f"Error on a block: {e}")

    if stop:
        break

    try:
      
        next_btn = driver.find_element(By.CSS_SELECTOR, "ul.pagination li.page-item a[rel='next']")
        if next_btn.is_enabled() and next_btn.is_displayed():
            next_btn.click()
            print("Next page")
            time.sleep(3) 
        else:
            print("No active next page")
            break
    except Exception:
        print("End of pagination")
        break

In [ ]:
# call for expressions of interest

try:
    driver.get("https://www.armp.cm/filtres?type=avis&val=12")
except WebDriverException as e:
    print(f"Error while loading the page: {e}")
    print("Waiting before retrying...")
    time.sleep(10) 
wait = WebDriverWait(driver, 10)

In [ ]:
# scraping


today = datetime.now()
stop = False

while True:
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li.list-group-item")))
    blocks = driver.find_elements(By.CSS_SELECTOR, "li.list-group-item")

    for i in range(len(blocks)):
        try:
            blocks = driver.find_elements(By.CSS_SELECTOR, "li.list-group-item")
            block = blocks[i]

            date_pub_text = block.find_element(By.XPATH, ".//div[contains(text(), 'Publié le')]/following-sibling::div").text.strip()

            date_pub = dateparser.parse(date_pub_text, settings={"DATE_ORDER": "DMY"})
            if not date_pub:
                continue
            
            if (today - date_pub).days > 30:
                print(f"Too old: {date_pub.strftime('%d-%m-%Y')}")
                stop = True
                break

            title = block.find_element(By.CSS_SELECTOR, "div.text-left strong").get_attribute("title").strip()
            publish_date = date_pub.strftime('%d-%m-%Y')
            link = block.find_element(By.XPATH, ".//a[contains(text(), 'Détails')]").get_attribute("href")

            if link:
                driver.get(link)
                time.sleep(2)
                html = driver.page_source
                soup = BeautifulSoup(html, "html.parser")

                expiration_date = None
                trs = soup.select("tbody tr")
                if len(trs) >= 2:
                    text = trs[1].get_text(separator=" ", strip=True).lower()
                    match = re.search(r"au plus tard(?: le)?\s+(\d{1,2}[/-]\d{1,2}[/-]\d{4})", text)
                    if match:
                        date_str = match.group(1).replace("/", "-")
                        try:
                            expiration_date = datetime.strptime(date_str, "%d-%m-%Y")
                        except:
                            pass

                if not expiration_date:
                    full_text = soup.get_text(separator=" ", strip=True).lower()
                    match = re.search(r"au plus tard(?: le)?\s+(\d{1,2}[/-]\d{1,2}[/-]\d{4})", full_text)
                    if not match:
                        match = re.search(r"before the\s+(\d{1,2}[/-]\d{1,2}[/-]\d{4})", full_text)
                    if match:
                        date_str = match.group(1).replace("/", "-")
                        try:
                            expiration_date = datetime.strptime(date_str, "%d-%m-%Y")
                        except:
                            pass

                tenders.append({
                        "title": title,
                        "publication_date": publish_date,
                        "expiration_date": expiration_date.strftime('%d-%m-%Y') if expiration_date else None,
                        "link": link,
                        "country": "Cameroon"
                    })
             
                driver.back()
                wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li.list-group-item")))

        except Exception as e:
            print(f"Error on a block: {e}")

    if stop:
        break

    try:
      
        next_btn = driver.find_element(By.CSS_SELECTOR, "ul.pagination li.page-item a[rel='next']")
        if next_btn.is_enabled() and next_btn.is_displayed():
            next_btn.click()
            print("Next page")
            time.sleep(3) 
        else:
            print("No active next page")
            break
    except Exception:
        print("End of pagination")
        break

In [ ]:
driver.quit()

In [ ]:
df = pd.DataFrame(tenders)
df.to_excel("../2_scraped_files/Cameroon.xlsx", index=False) # modify